In [1]:
import sys
sys.path.append("..") 
from Dataset.DatasetClass import quarkGluonEvent
from Task1.calculate_std_mean import calculate_std_mean
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch
from torchvision import transforms
import pickle
from train import train_autoencoder
from autoencoder import AutoEncoder2D
import torchsummary

In [2]:
import random 
import numpy as np

def set_all_seeds(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    g = torch.Generator()
    g.manual_seed(seed)  
    
    return g

In [3]:
DATA_PATH = "../Dataset/Data/quark-gluon_data-set_n139306.hdf5"
g = set_all_seeds(42)
VAL_SPLIT = 0.20
TEST_SPLIT = 0.10
BATCH_SIZE = 256
NUM_WORKERS = 6

In [4]:
dataset = quarkGluonEvent(DATA_PATH)
n = len(dataset)

n = len(dataset)

train_size = int(n * (1 - (VAL_SPLIT + TEST_SPLIT)))
val_size   = int(n * VAL_SPLIT)
test_size  = n - train_size - val_size  

train_dataset, _, _ = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=g
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g)

Calculation of mean and std for normalization

In [5]:
mean, std = calculate_std_mean(train_loader)

Batches:   0%|          | 0/381 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
with open('mean_std.pkl', 'rb') as f:
    mean, std = pickle.load(f)

In [ ]:
transform = transforms.Compose([
    transforms.Normalize(mean=mean, std=std),
])

NameError: name 'transforms' is not defined

In [5]:
del train_loader, train_dataset

dataset = quarkGluonEvent(DATA_PATH, transform=transform)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=g
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, generator=g)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, generator=g)

NameError: name 'transform' is not defined

In [6]:
input_shape = dataset[0][0].shape
print(f"Input shape: {input_shape}")

model = AutoEncoder2D(input_shape=input_shape, latent_dim=128, base=32, layer_count=3)

torchsummary.summary(model, input_size=input_shape, device='cpu')

Input shape: torch.Size([3, 125, 125])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 63, 63]             864
       BatchNorm2d-2           [-1, 32, 63, 63]              64
              ReLU-3           [-1, 32, 63, 63]               0
            Conv2d-4           [-1, 64, 32, 32]          18,432
       BatchNorm2d-5           [-1, 64, 32, 32]             128
              ReLU-6           [-1, 64, 32, 32]               0
            Conv2d-7          [-1, 128, 16, 16]          73,728
       BatchNorm2d-8          [-1, 128, 16, 16]             256
              ReLU-9          [-1, 128, 16, 16]               0
           Linear-10                  [-1, 128]       4,194,432
        Encoder2D-11                  [-1, 128]               0
           Linear-12                [-1, 32768]       4,227,072
  ConvTranspose2d-13           [-1, 64, 32, 32]         131,072
